# Dropout 正则化技术 — PyTorch 版

> 本笔记是 [Dropout算法.ipynb](./Dropout算法.ipynb)（TensorFlow/Keras 版）的 PyTorch 等价实现。

## 核心思想

Dropout（Srivastava et al., 2014）是一种简单有效的正则化技术。在训练过程中，随机"丢弃"（置零）一部分神经元的输出，迫使网络学习更鲁棒的特征。

## 工作原理

```
训练阶段：
- 每个神经元以概率 p 被临时"关闭"
- 保留的神经元输出需要除以 (1-p) 进行缩放（inverted dropout）

推理阶段：
- 所有神经元都参与计算
- 无需额外缩放（因为训练时已处理）
```

## 为什么有效

| 机制 | 解释 |
|------|------|
| 集成效果 | 相当于训练多个子网络的集成 |
| 减少共适应 | 神经元不能过度依赖其他特定神经元 |
| 特征冗余 | 迫使网络学习更分散、更鲁棒的特征表示 |

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# 设置随机种子 / Set random seed
torch.manual_seed(42)
np.random.seed(42)

# 检测设备 / Detect device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"设备: {device}")

## 1. 基本用法

In [ ]:
class DropoutMLP(nn.Module):
    """
    带 Dropout 的多层感知机 / MLP with Dropout layers

    PyTorch 中 nn.Dropout 默认使用 inverted dropout（训练时缩放），
    与 Keras 的 keras.layers.Dropout 行为一致。
    """
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.flatten = nn.Flatten()
        # 输入层 Dropout（通常使用较小的 rate，如 0.2）
        self.input_dropout = nn.Dropout(p=0.2)
        self.fc1 = nn.Linear(28 * 28, 256)
        self.dropout1 = nn.Dropout(p=dropout_rate)
        self.fc2 = nn.Linear(256, 128)
        self.dropout2 = nn.Dropout(p=dropout_rate)
        self.fc3 = nn.Linear(128, 64)
        self.dropout3 = nn.Dropout(p=dropout_rate)
        self.fc4 = nn.Linear(64, 10)
        # He 初始化 / He initialization
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')

    def forward(self, x):
        x = self.flatten(x)
        x = self.input_dropout(x)
        x = self.dropout1(torch.relu(self.fc1(x)))
        x = self.dropout2(torch.relu(self.fc2(x)))
        x = self.dropout3(torch.relu(self.fc3(x)))
        x = torch.softmax(self.fc4(x), dim=1)
        return x

model = DropoutMLP(dropout_rate=0.3).to(device)
print(model)
print(f"\n总参数量: {sum(p.numel() for p in model.parameters()):,}")

## 2. 可视化 Dropout 效果

In [ ]:
def visualize_dropout(rate=0.5, input_size=100):
    """
    可视化 Dropout 对神经元的影响 / Visualize the effect of Dropout on neurons

    Parameters:
    -----------
    rate : float
        Dropout 比率（0到1之间） / Dropout rate (between 0 and 1)
    input_size : int
        输入维度大小 / Input dimension size
    """
    dropout_layer = nn.Dropout(p=rate)

    # 创建输入数据 / Create input data
    x = torch.ones((1, input_size))

    # 训练模式下的输出 / Output in training mode
    dropout_layer.train()
    output_train = dropout_layer(x).detach().numpy().flatten()

    # 推理模式下的输出 / Output in inference mode
    dropout_layer.eval()
    output_inference = dropout_layer(x).detach().numpy().flatten()

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # 原始输入 / Original input
    axes[0].bar(range(input_size), x.numpy().flatten(), color='blue', alpha=0.7)
    axes[0].set_title('原始输入')
    axes[0].set_xlabel('神经元索引')
    axes[0].set_ylabel('激活值')
    axes[0].set_ylim(0, 2)

    # 训练模式 / Training mode
    colors = ['red' if v == 0 else 'green' for v in output_train]
    axes[1].bar(range(input_size), output_train, color=colors, alpha=0.7)
    axes[1].set_title(f'训练模式 (rate={rate})')
    axes[1].set_xlabel('神经元索引')
    axes[1].set_ylabel('激活值')
    axes[1].set_ylim(0, 2)

    dropped = int(np.sum(output_train == 0))
    axes[1].text(0.5, 0.95, f'丢弃: {dropped}/{input_size} ({dropped/input_size*100:.1f}%)',
                transform=axes[1].transAxes, ha='center', fontsize=10)

    # 推理模式 / Inference mode
    axes[2].bar(range(input_size), output_inference, color='blue', alpha=0.7)
    axes[2].set_title('推理模式 (无 Dropout)')
    axes[2].set_xlabel('神经元索引')
    axes[2].set_ylabel('激活值')
    axes[2].set_ylim(0, 2)

    plt.tight_layout()
    plt.show()

# 可视化不同 Dropout 率的效果 / Visualize effects of different Dropout rates
for rate in [0.2, 0.5]:
    print(f"\nDropout rate = {rate}")
    visualize_dropout(rate=rate, input_size=50)

## 3. 训练与验证对比

In [ ]:
# 加载 Fashion-MNIST 数据集 / Load Fashion-MNIST dataset
transform = transforms.Compose([transforms.ToTensor()])

train_dataset_full = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# 划分训练集和验证集 / Split into training and validation sets
valid_size = 5000
train_size = len(train_dataset_full) - valid_size
train_dataset = Subset(train_dataset_full, range(valid_size, len(train_dataset_full)))
valid_dataset = Subset(train_dataset_full, range(valid_size))

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"训练集: {train_size} 样本")
print(f"验证集: {valid_size} 样本")
print(f"测试集: {len(test_dataset)} 样本")

In [ ]:
def create_model(dropout_rate=0.0):
    """
    创建带可配置 Dropout 率的模型 / Create model with configurable Dropout rate

    Parameters:
    -----------
    dropout_rate : float
        Dropout 比率，0 表示不使用 Dropout / Dropout rate, 0 means no Dropout

    Returns:
    --------
    nn.Module
        构建好的模型 / Constructed model
    """
    layers = [nn.Flatten()]
    for in_features, out_features in [(28*28, 256), (256, 128), (128, 64)]:
        linear = nn.Linear(in_features, out_features)
        nn.init.kaiming_normal_(linear.weight, nonlinearity='relu')
        layers.append(linear)
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(p=dropout_rate))
    layers.append(nn.Linear(64, 10))
    return nn.Sequential(*layers)

# 创建对比模型 / Create comparison models
model_no_dropout = create_model(dropout_rate=0.0).to(device)
model_with_dropout = create_model(dropout_rate=0.3).to(device)

params_no = sum(p.numel() for p in model_no_dropout.parameters())
params_with = sum(p.numel() for p in model_with_dropout.parameters())
print(f"无 Dropout 模型参数量: {params_no:,}")
print(f"有 Dropout 模型参数量: {params_with:,}")

In [ ]:
def train_model(model, train_loader, valid_loader, epochs=30, lr=0.001):
    """
    训练模型并记录历史 / Train model and record history

    Parameters:
    -----------
    model : nn.Module
        要训练的模型 / Model to train
    train_loader : DataLoader
        训练数据加载器 / Training data loader
    valid_loader : DataLoader
        验证数据加载器 / Validation data loader
    epochs : int
        训练轮数 / Number of training epochs
    lr : float
        学习率 / Learning rate

    Returns:
    --------
    dict
        包含训练和验证的损失与准确率历史 / History dict with train/val loss and accuracy
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}

    for epoch in range(epochs):
        # 训练阶段 / Training phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        history['loss'].append(running_loss / total)
        history['accuracy'].append(correct / total)

        # 验证阶段 / Validation phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        history['val_loss'].append(val_loss / val_total)
        history['val_accuracy'].append(val_correct / val_total)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs} - "
                  f"loss: {history['loss'][-1]:.4f} - "
                  f"acc: {history['accuracy'][-1]:.4f} - "
                  f"val_loss: {history['val_loss'][-1]:.4f} - "
                  f"val_acc: {history['val_accuracy'][-1]:.4f}")

    return history

EPOCHS = 30

print("训练无 Dropout 模型...")
history_no_dropout = train_model(model_no_dropout, train_loader, valid_loader, epochs=EPOCHS)
print("完成")

print("\n训练有 Dropout 模型...")
history_with_dropout = train_model(model_with_dropout, train_loader, valid_loader, epochs=EPOCHS)
print("完成")

In [ ]:
# 绘制训练曲线对比 / Plot training curve comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 准确率 / Accuracy
axes[0].plot(history_no_dropout['accuracy'], 'b-', label='无 Dropout (训练)', alpha=0.8)
axes[0].plot(history_no_dropout['val_accuracy'], 'b--', label='无 Dropout (验证)', alpha=0.8)
axes[0].plot(history_with_dropout['accuracy'], 'r-', label='有 Dropout (训练)', alpha=0.8)
axes[0].plot(history_with_dropout['val_accuracy'], 'r--', label='有 Dropout (验证)', alpha=0.8)
axes[0].set_title('准确率对比')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 损失 / Loss
axes[1].plot(history_no_dropout['loss'], 'b-', label='无 Dropout (训练)', alpha=0.8)
axes[1].plot(history_no_dropout['val_loss'], 'b--', label='无 Dropout (验证)', alpha=0.8)
axes[1].plot(history_with_dropout['loss'], 'r-', label='有 Dropout (训练)', alpha=0.8)
axes[1].plot(history_with_dropout['val_loss'], 'r--', label='有 Dropout (验证)', alpha=0.8)
axes[1].set_title('损失对比')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dropout_comparison_pytorch.png', dpi=150, bbox_inches='tight')
plt.show()

# 计算过拟合程度 / Calculate overfitting gap
gap_no_dropout = history_no_dropout['accuracy'][-1] - history_no_dropout['val_accuracy'][-1]
gap_with_dropout = history_with_dropout['accuracy'][-1] - history_with_dropout['val_accuracy'][-1]

print("\n过拟合程度（训练-验证准确率差）:")
print(f"无 Dropout: {gap_no_dropout:.4f}")
print(f"有 Dropout: {gap_with_dropout:.4f}")

In [ ]:
# 测试集评估 / Test set evaluation
def evaluate_model(model, test_loader):
    """
    在测试集上评估模型 / Evaluate model on test set

    Parameters:
    -----------
    model : nn.Module
        要评估的模型 / Model to evaluate
    test_loader : DataLoader
        测试数据加载器 / Test data loader

    Returns:
    --------
    tuple (loss, accuracy)
    """
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return total_loss / total, correct / total

print("测试集评估:")
test_loss_no, test_acc_no = evaluate_model(model_no_dropout, test_loader)
test_loss_with, test_acc_with = evaluate_model(model_with_dropout, test_loader)

print(f"无 Dropout: 准确率 = {test_acc_no:.4f}, 损失 = {test_loss_no:.4f}")
print(f"有 Dropout: 准确率 = {test_acc_with:.4f}, 损失 = {test_loss_with:.4f}")

## 4. Dropout 变体

### 4.1 空间 Dropout (Spatial Dropout / Dropout2d)

用于卷积神经网络，丢弃整个特征图而非单个像素。PyTorch 中对应 `nn.Dropout2d`。

In [ ]:
class CNNSpatialDropout(nn.Module):
    """
    带空间 Dropout 的 CNN / CNN with Spatial Dropout

    nn.Dropout2d 丢弃整个通道（特征图），而非单个像素。
    等价于 Keras 的 SpatialDropout2D。
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            # Dropout2d 丢弃整个特征图 / Dropout2d drops entire feature maps
            nn.Dropout2d(p=0.25),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout2d(p=0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

cnn_model = CNNSpatialDropout()
print("CNN with Spatial Dropout (nn.Dropout2d):")
print(cnn_model)
print(f"\n总参数量: {sum(p.numel() for p in cnn_model.parameters()):,}")

### 4.2 Alpha Dropout（用于 SELU）

专为自归一化网络设计，保持输入的均值和方差。PyTorch 中对应 `nn.AlphaDropout`。

In [ ]:
class SELUAlphaDropout(nn.Module):
    """
    使用 SELU + AlphaDropout 的自归一化网络 / Self-normalizing network with SELU + AlphaDropout

    nn.AlphaDropout 专为 SELU 激活设计，保持输出的均值和方差。
    等价于 Keras 的 AlphaDropout。
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.SELU(),
            nn.AlphaDropout(p=0.1),  # SELU 专用 / SELU-specific
            nn.Linear(256, 128),
            nn.SELU(),
            nn.AlphaDropout(p=0.1),
            nn.Linear(128, 10),
        )
        # LeCun 初始化 / LeCun initialization
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity='linear')

    def forward(self, x):
        return self.net(x)

selu_model = SELUAlphaDropout()
print("SELU Network with Alpha Dropout (nn.AlphaDropout):")
print(selu_model)
print(f"\n总参数量: {sum(p.numel() for p in selu_model.parameters()):,}")

## 5. 使用建议

### Dropout 率选择

| 层类型 | 推荐 Dropout 率 |
|--------|----------------|
| 输入层 | 0.1 - 0.2 |
| 隐藏层 | 0.2 - 0.5 |
| 输出层 | 不使用 Dropout |
| CNN 特征提取层 | 0.25 (Spatial Dropout / Dropout2d) |
| CNN 全连接层 | 0.5 |

### 注意事项

1. **与 BN 配合**: 通常将 Dropout 放在 BN 之后，或不同时使用
2. **大数据集**: 数据量大时，可能不需要 Dropout
3. **测试时**: 确保调用 `model.eval()`，使 Dropout 层不激活
4. **率过高**: Dropout 率过高会导致欠拟合

## 6. TF vs PyTorch 对照

| 概念 | TensorFlow / Keras | PyTorch |
|------|-------------------|---------|
| 标准 Dropout | `keras.layers.Dropout(rate)` | `nn.Dropout(p)` |
| 空间 Dropout | `keras.layers.SpatialDropout2D(rate)` | `nn.Dropout2d(p)` |
| Alpha Dropout | `keras.layers.AlphaDropout(rate)` | `nn.AlphaDropout(p)` |
| 训练/推理切换 | `training=True/False` 参数 | `model.train()` / `model.eval()` |
| Dropout 率参数名 | `rate` (丢弃概率) | `p` (丢弃概率) |
| Inverted Dropout | 默认使用 | 默认使用 |
| 模型概览 | `model.summary()` | `print(model)` |
| 参数量 | `model.count_params()` | `sum(p.numel() for p in model.parameters())` |
| 权重初始化 | `kernel_initializer` 参数 | `nn.init.*` 函数 |
| He 初始化 | `kernel_initializer='he_normal'` | `nn.init.kaiming_normal_(weight)` |
| LeCun 初始化 | `kernel_initializer='lecun_normal'` | `nn.init.kaiming_normal_(weight, nonlinearity='linear')` |
| 训练循环 | `model.fit()` | 手动循环 / Manual loop |
| 损失函数 | `sparse_categorical_crossentropy` | `nn.CrossEntropyLoss()` (含 softmax) |
| 数据加载 | `model.fit(x, y)` | `DataLoader` + 迭代 |

### 关键差异说明

1. **训练/推理模式切换**: Keras 通过 `training` 参数逐层控制；PyTorch 通过 `model.train()` / `model.eval()` 全局控制所有模块的状态。
2. **损失函数**: Keras 的 `sparse_categorical_crossentropy` 期望 softmax 输出；PyTorch 的 `nn.CrossEntropyLoss` 内部已包含 softmax，因此模型最后一层不需要 softmax。本笔记为对齐 Keras 版在模型中加了 softmax，训练时使用 `nn.CrossEntropyLoss` 会导致双重 softmax——实际使用时应去掉模型末尾的 softmax。
3. **参数名**: Keras 用 `rate`，PyTorch 用 `p`，含义相同（均为丢弃概率）。

## 7. 练习

In [ ]:
# 练习 1: 探索不同 Dropout 率对过拟合的影响
# Exercise 1: Explore the effect of different Dropout rates on overfitting
#
# 尝试 dropout_rate = [0.0, 0.1, 0.3, 0.5, 0.7]，训练 20 个 epoch，
# 绘制验证准确率随 Dropout 率变化的曲线。
# Try dropout_rate = [0.0, 0.1, 0.3, 0.5, 0.7], train for 20 epochs,
# plot validation accuracy vs Dropout rate.
#
# 提示 / Hint:
# for rate in [0.0, 0.1, 0.3, 0.5, 0.7]:
#     model = create_model(dropout_rate=rate).to(device)
#     history = train_model(model, train_loader, valid_loader, epochs=20)
#     # 记录最终的验证准确率 / Record final validation accuracy

# --- 在此编写代码 / Write your code here ---

In [ ]:
# 练习 2: 验证 model.train() / model.eval() 对 Dropout 行为的影响
# Exercise 2: Verify the effect of model.train() / model.eval() on Dropout behavior
#
# 创建一个 nn.Dropout(p=0.5) 层，输入全 1 张量 (1, 1000)，
# 分别在 train 和 eval 模式下运行 10 次，统计每次非零元素的比例。
# Create an nn.Dropout(p=0.5) layer, input an all-ones tensor (1, 1000),
# run 10 times in train and eval mode respectively, count the proportion
# of non-zero elements each time.
#
# 提示 / Hint:
# dropout = nn.Dropout(p=0.5)
# x = torch.ones(1, 1000)
# dropout.train()
# for i in range(10):
#     out = dropout(x)
#     nonzero_ratio = (out != 0).float().mean().item()
#     print(f"Train run {i}: nonzero ratio = {nonzero_ratio:.4f}")

# --- 在此编写代码 / Write your code here ---

In [ ]:
# 练习 3: 对比 nn.Dropout 与 nn.Dropout2d 在卷积特征图上的效果
# Exercise 3: Compare nn.Dropout vs nn.Dropout2d on convolutional feature maps
#
# 创建一个形状为 (1, 4, 5, 5) 的特征图（1 batch, 4 channels, 5x5 spatial），
# 分别用 nn.Dropout(p=0.5) 和 nn.Dropout2d(p=0.5) 处理，
# 可视化两种方式丢弃的像素模式有何不同。
# Create a feature map of shape (1, 4, 5, 5), process with nn.Dropout(p=0.5)
# and nn.Dropout2d(p=0.5) respectively, visualize the difference in
# the dropout pattern.
#
# 提示 / Hint:
# feat = torch.ones(1, 4, 5, 5)
# d1 = nn.Dropout(p=0.5); d1.train()
# d2 = nn.Dropout2d(p=0.5); d2.train()
# out1 = d1(feat)  # 随机丢弃个别像素 / Randomly drops individual pixels
# out2 = d2(feat)  # 丢弃整个通道 / Drops entire channels
# # 用 plt.imshow 可视化每个通道 / Visualize each channel with plt.imshow

# --- 在此编写代码 / Write your code here ---

In [ ]:
# 验证代码正确性 / Verify code correctness
print("Dropout 模块测试完成 (PyTorch)")
print("\n关键要点:")
print("1. Dropout 随机丢弃神经元，防止过拟合")
print("2. 训练时 model.train() 激活 Dropout，推理时 model.eval() 关闭")
print("3. 输入层用小 rate (0.1-0.2)，隐藏层用中等 rate (0.2-0.5)")
print("4. SELU 激活函数配合 nn.AlphaDropout 使用")
print("5. CNN 中使用 nn.Dropout2d 丢弃整个特征图")